<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_2/lessons/lesson_22_practicum_recursion/note_lesson_22_recursion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Урок 22 — Практикум П4: рекурсія, «розділяй і володарюй», перебір з поверненням

Три задачі сервісу «Смачно + Таксі», у яких відповідь для цілого збирається з відповідей для частин:

1. **Вкладене меню** невідомої глибини: скільки страв, де «Узвар», скільки коштує все меню.
2. **Сортування доставок**: вставка з П1 — `O(n²)`, злиття — `O(n log n)`.
3. **Ваучер на будь-яку кількість поїздок**: Two Sum з П3 шукав пару, а тепер потрібні набори будь-якого розміру.

Виконуй клітинки **зверху вниз**; перед **Прогнозом** спершу скажи, що буде. Теорія, дерева рішень і «рекурсія чи цикл» — у книзі: [Урок 22. Практикум П4](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m2/lesson_22/).

## 🔁 Пригадай (без підглядання)

1. Скільки кроків бінарного пошуку на 1 000 000 записів?
2. Що знаходив Two Sum у П3?
3. Що робить `@lru_cache`?

<details>
<summary>Відповіді</summary>

1. Близько 20.
2. Пару чисел із заданою сумою за один прохід.
3. Запам'ятовує результати функції.

</details>

## 1. Рекурсія на вкладеному меню

**Прогноз:** скільки страв у меню? (рахуй лише листки з цінами)

In [ ]:
MENU = {
    "Кухня": {
        "Перші страви": {"Борщ": 95, "Юшка": 85},
        "Основні": {"Вареники": 80, "Деруни": 75},
    },
    "Напої": {
        "Гарячі": {"Чай": 30, "Кава": {"Еспресо": 40, "Лате": 55}},
        "Холодні": {"Узвар": 35},
    },
    "Хліб": 10,
}


def count_dishes(node):
    if not isinstance(node, dict):
        return 1
    return sum(count_dishes(child) for child in node.values())


print(count_dishes(MENU))
print(count_dishes(MENU["Напої"]))
print(count_dishes(95))

<details>
<summary>Відповідь</summary>

9. Базовий випадок — страва (1), рекурсивний — категорія (сума по частинах).

</details>

In [ ]:
def find_path(node, dish, path=()):
    if not isinstance(node, dict):
        return None
    for name, child in node.items():
        if name == dish:
            return path + (name,)
        found = find_path(child, dish, path + (name,))
        if found:
            return found
    return None


print(find_path(MENU, "Узвар"))
print(find_path(MENU, "Лате"))
print(find_path(MENU, "Піца"))

### Межа глибини

**Прогноз:** чи впаде `countdown(100_000)`?

In [ ]:
import sys


def countdown(n):
    if n == 0:
        return 0
    return countdown(n - 1)


print(sys.getrecursionlimit())
print(countdown(500))
try:
    countdown(100_000)
except RecursionError as error:
    print(type(error).__name__)

<details>
<summary>Відповідь</summary>

Так, `RecursionError`: Python обмежує глибину стеку ~1000 кадрів. Для лінійних даних — цикл.

</details>

## 🛠 Вправа 1. Генератор страв зі шляхами

`dishes(node, path=())` — рекурсивний генератор пар `(шлях, ціна)`. Використай `yield from`.

In [ ]:
def dishes(node, path=()):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    if not isinstance(node, dict):
        yield path, node
        return
    for name, child in node.items():
        yield from dishes(child, path + (name,))
    # END SOLUTION


print(next(dishes(MENU)))
assert next(dishes(MENU)) == (("Кухня", "Перші страви", "Борщ"), 95)
assert len(list(dishes(MENU))) == 9
assert (("Напої", "Гарячі", "Кава", "Лате"), 55) in list(dishes(MENU))
assert sum(1 for _ in dishes(MENU)) == count_dishes(MENU)
print("✅ Вправа 1 пройдена")

## 🛠 Вправа 2. Звіт за меню

Напиши за тим самим шаблоном `total_price`, `cheapest`, `depth`.

In [ ]:
def total_price(node):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    if not isinstance(node, dict):
        return node
    return sum(total_price(child) for child in node.values())
    # END SOLUTION


def cheapest(node):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    if not isinstance(node, dict):
        return node
    return min(cheapest(child) for child in node.values())
    # END SOLUTION


def depth(node):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    if not isinstance(node, dict):
        return 0
    return 1 + max(depth(child) for child in node.values())
    # END SOLUTION


assert total_price(MENU) == 505 and cheapest(MENU["Кухня"]) == 75 and depth(MENU) == 4
assert cheapest(MENU) == 10 and depth(95) == 0
print("✅ Вправа 2 пройдена")

## 2. Розділяй і володарюй: сортування злиттям

In [ ]:
def merge(left, right, counter):
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        counter[0] += 1
        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1
    return result + left[i:] + right[j:]


def merge_sort(items, counter):
    if len(items) <= 1:
        return list(items)
    middle = len(items) // 2
    left = merge_sort(items[:middle], counter)
    right = merge_sort(items[middle:], counter)
    return merge(left, right, counter)


fares = [230, 150, 270, 180, 320, 120, 410, 150]
counter = [0]
print(merge_sort(fares, counter), counter[0])

**Прогноз:** у скільки разів зростуть порівняння вставки й злиття, коли даних удвічі більше?

In [ ]:
def insertion_sort(items, counter):
    result = list(items)
    for k in range(1, len(result)):
        j = k
        while j > 0:
            counter[0] += 1
            if result[j - 1] <= result[j]:
                break
            result[j - 1], result[j] = result[j], result[j - 1]
            j -= 1
    return result


for n in [1000, 2000, 4000]:
    data = list(range(n, 0, -1))
    slow, fast = [0], [0]
    insertion_sort(data, slow)
    merge_sort(data, fast)
    print(n, slow[0], fast[0])

<details>
<summary>Відповідь</summary>

Вставка — ×4 (`O(n²)`), злиття — трохи більше ніж ×2 (`O(n log n)`).

</details>

## 🛠 Вправа 3. Бінарний пошук рекурсією

`binary_search(items, target, low=0, high=None)` — повертає індекс або `-1`. База: порожній проміжок. Рекурсія — в одну половину.

In [ ]:
def binary_search(items, target, low=0, high=None):
    if high is None:
        high = len(items)
    # YOUR CODE HERE
    # BEGIN SOLUTION
    if low >= high:
        return -1
    middle = (low + high) // 2
    if items[middle] == target:
        return middle
    if items[middle] < target:
        return binary_search(items, target, middle + 1, high)
    return binary_search(items, target, low, middle)
    # END SOLUTION


starts = [425, 510, 612, 700, 845, 930, 1035, 1120, 1210, 1290, 1375]
assert binary_search(starts, 1120) == 7
assert binary_search(starts, 425) == 0 and binary_search(starts, 1375) == 10
assert binary_search(starts, 999) == -1 and binary_search([], 5) == -1
print("✅ Вправа 3 пройдена")

## 3. Перебір з поверненням: ваучер на будь-яку кількість поїздок

In [ ]:
def voucher_sets(fares, target):
    found = []
    calls = [0]

    def explore(start, chosen, total):
        calls[0] += 1
        if total == target:
            found.append(list(chosen))
            return
        for i in range(start, len(fares)):
            chosen.append(fares[i])
            explore(i + 1, chosen, total + fares[i])
            chosen.pop()

    explore(0, [], 0)
    return found, calls[0]


trip_fares = [230, 150, 270, 180, 120, 410]
print(voucher_sets(trip_fares, 500))

**Прогноз:** у скільки разів менше викликів дасть відсікання на 12 поїздках?

In [ ]:
def voucher_sets_pruned(fares, target):
    fares = sorted(fares)
    found = []
    calls = [0]

    def explore(start, chosen, total):
        calls[0] += 1
        if total == target:
            found.append(list(chosen))
            return
        for i in range(start, len(fares)):
            if total + fares[i] > target:
                break
            chosen.append(fares[i])
            explore(i + 1, chosen, total + fares[i])
            chosen.pop()

    explore(0, [], 0)
    return found, calls[0]


print(voucher_sets_pruned(trip_fares, 500))

big_shift = [230, 150, 270, 180, 120, 410, 95, 60, 310, 200, 140, 175]
print(voucher_sets(big_shift, 500)[1], voucher_sets_pruned(big_shift, 500)[1])

<details>
<summary>Відповідь</summary>

3454 проти 150 — більш ніж у 20 разів. Набори ті самі.

</details>

## 🛠 Вправа 4. Обід рівно на 150 грн

`lunch_sets(menu, budget)` — усі набори назв страв (кожна не більше разу) на рівно `budget`. Страви бери генератором `dishes` у порядку меню; перебір з поверненням з відсіканням гілок, де сума вже більша за бюджет.

In [ ]:
def lunch_sets(menu, budget):
    items = [(path[-1], price) for path, price in dishes(menu)]
    found = []
    # YOUR CODE HERE
    # BEGIN SOLUTION
    def explore(start, chosen, total):
        if total == budget:
            found.append([name for name, _ in chosen])
            return
        for i in range(start, len(items)):
            name, price = items[i]
            if total + price > budget:
                continue
            chosen.append(items[i])
            explore(i + 1, chosen, total + price)
            chosen.pop()

    explore(0, [], 0)
    # END SOLUTION
    return found


result = lunch_sets(MENU, 150)
for names in result:
    print(names)
assert result == [["Борщ", "Лате"], ["Юшка", "Чай", "Узвар"], ["Юшка", "Лате", "Хліб"],
                  ["Вареники", "Чай", "Еспресо"], ["Деруни", "Чай", "Узвар", "Хліб"],
                  ["Деруни", "Еспресо", "Узвар"]]
assert len(lunch_sets(MENU, 200)) == 10 and lunch_sets(MENU, 5) == []
print("✅ Вправа 4 пройдена")

## 4. Коли рекурсія повторює роботу

**Прогноз:** скільки викликів зробить `fib(20)`?

In [ ]:
calls = [0]


def fib(n):
    calls[0] += 1
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)


print(fib(20), calls[0])


from functools import lru_cache


@lru_cache(maxsize=None)
def fib_memo(n):
    if n < 2:
        return n
    return fib_memo(n - 1) + fib_memo(n - 2)


print(fib_memo(20), fib_memo.cache_info().misses)

<details>
<summary>Відповідь</summary>

21 891 без кешу і 21 обчислення з `lru_cache`: ті самі `fib(k)` рахувались знову й знову. Далі — динамічне програмування (урок 26).

</details>

## 5. Рекурсія чи цикл: явний стек

In [ ]:
def count_dishes_iterative(menu):
    stack = [menu]
    count = 0
    while stack:
        node = stack.pop()
        if isinstance(node, dict):
            stack.extend(node.values())
        else:
            count += 1
    return count


print(count_dishes_iterative(MENU))

## ✅ Самоперевірка

1. Що буде без базового випадку?
2. Звідки `log n` у сортуванні злиттям?
3. Що робить `chosen.pop()` у переборі з поверненням?
4. Чому відсікання `break` правильне лише для відсортованого списку?
5. Що змінює `lru_cache` для `fib`?

<details>
<summary>Відповіді</summary>

1. `RecursionError`: функція викликає себе, доки не закінчиться стек.
2. Поділ навпіл дає `log₂ n` рівнів; на кожному злиття проходить n елементів.
3. Прибирає останній вибір, щоб спробувати інший на його місці.
4. «Наступні теж не влізуть» — правда, лише якщо наступні не менші за поточний.
5. Кожне `fib(k)` рахується один раз: 21 обчислення замість 21 891.

</details>

### Шпаргалка

```python
def solve(node):                   # рекурсія
    if базовий_випадок(node):
        return відповідь
    return згорнути(solve(частина) for частина in частини(node))

def merge_sort(items):             # розділяй і володарюй
    if len(items) <= 1: return items
    mid = len(items) // 2
    return merge(merge_sort(items[:mid]), merge_sort(items[mid:]))

def explore(start, chosen, total): # перебір з поверненням
    if total == target: found.append(list(chosen)); return
    for i in range(start, len(items)):
        if total + items[i] > target: break      # відсікання (відсортовано)
        chosen.append(items[i]); explore(i + 1, chosen, total + items[i]); chosen.pop()

@lru_cache(maxsize=None)           # підзадачі повторюються
```

## Далі

- **Урок 23 — `@property`, декоратори класів, dunder**: `sorted(deliveries)` і `len(menu)` для наших класів.
- **Практикум П5 (урок 26)**: динамічне програмування.